In [61]:
import pandas as pd
import numpy as np
from nbconvert import export


In [62]:
#Loading in the roads data file
df1 = pd.read_csv('./data/_roads3.csv')

# print(df1.head())

#Filtering on only the N1 road
roads = ['N1', 'N2'] + [f'N{i}' for i in range(100, 300)]

dfN = df1[df1['road'].isin(roads)]
# dfN['road'].value_counts()


In [63]:
# import matplotlib.pyplot as plt
#
# df_plot = dfN[dfN['road'].isin(['N1', 'N2'])]
#
# for road, data in df_plot.groupby('road'):
#     plt.plot(data['lon'], data['lat'], label=road)
#
# plt.legend()
# plt.show()

In [64]:
#Creating the source and sink for the new data frame, based on the minimal and maximal chainage
source_rows = dfN.loc[dfN.groupby('road')['chainage'].idxmin()]
sink_rows = dfN.loc[dfN.groupby('road')['chainage'].idxmax()]

df_proc = (
    pd.concat([source_rows, sink_rows])
    .drop_duplicates(subset=['road', 'chainage', 'lrp'])
    .sort_values(['road', 'chainage'])
    .reset_index(drop=True)
)

df_proc


,road,chainage,lrp,lat,lon,gap,type,name
0,N1,0.000,LRPS,23.706028,90.443333,NaN,Others,Start of Road after Jatrabari Flyover infront...
1,N1,462.254,LRPE,20.862917,92.298083,NaN,Others,"End of Road at Shapla Chattar ,Teknaf Meet wit..."
2,N101,0.000,LRPS,23.454139,91.212861,NaN,Others,Start of Road from N120 at Balutopa
3,N101,6.021,LRPE,23.459306,91.253389,NaN,Others,End of road Bibir bazar Bridge
4,N102,0.000,LRPS,23.478972,91.118194,NaN,Others,Start of road from N1 Mainamati
5,N102,82.682,LRPE,24.050611,91.114667,NaN,Others,Meet with N 2 at Sanail
6,N103,0.000,LRPS,23.957028,91.115528,NaN,Others,Road start from N102 at Kuatali.
7,N103,4.779,LRPE,23.996889,91.109278,NaN,Others,Intersection with N102 at Ghaturia.
8,N104,0.000,LRPS,23.009667,91.399416,NaN,Others,Intersection with Z1031
9,N104,49.630,LRPE,22.825749,91.101444,NaN,Others,Meet with Z1441& Z1405 at Sonapur.


In [71]:
df_proc['total_length'] = df_proc.groupby('road')['chainage'].transform('max')

df_filtered = df_proc[df_proc['total_length'] > 25]
df_filtered.head()
df_filtered.road.value_counts()

road
N1      2
N102    2
N104    2
N105    2
N106    2
N2      2
N204    2
N207    2
N208    2
Name: count, dtype: int64

In [50]:
# import matplotlib.pyplot as plt
#
# for road, data in dfN.groupby('road'):
#     plt.plot(data['lon'], data['lat'], label=road)
#
# plt.legend()
# plt.show()

In [37]:
#Loading in the Bridges data
df_BMMS = pd.read_excel('./data/BMMS_overview.xlsx')
#Selecting the columns that we need
cols = [
    "road",
    "name",
    "LRPName",
    "length",
    "chainage",
    "lat",
    "lon",
    "condition",
]

bridges_BMMS = df_BMMS[cols].copy()
bridges_BMMS = bridges_BMMS[bridges_BMMS['road'].isin(['N1', 'N2'])]
bridges_BMMS.head(20)
#Preprocessing names of bridges so they are more uniform for later duplicates deletion
bridges_BMMS['name_clean'] = bridges_BMMS['name'].astype(str).str.replace(' ', '').str.upper()
#Rounding the location of bridges in order to delete duplicates that are really close together later
bridges_BMMS['lat_round'] = bridges_BMMS['lat'].round(4)
bridges_BMMS['lon_round'] = bridges_BMMS['lon'].round(4)
bridges_BMMS.info()



<class 'pandas.core.frame.DataFrame'>
Index: 1347 entries, 0 to 19385
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   road        1347 non-null   object 
 1   name        1347 non-null   object 
 2   LRPName     1347 non-null   object 
 3   length      1346 non-null   float64
 4   chainage    1347 non-null   float64
 5   lat         1347 non-null   float64
 6   lon         1347 non-null   float64
 7   condition   1347 non-null   object 
 8   name_clean  1347 non-null   object 
 9   lat_round   1347 non-null   float64
 10  lon_round   1347 non-null   float64
dtypes: float64(6), object(5)
memory usage: 126.3+ KB


In [ ]:
#Dropping duplicates
df_unique = bridges_BMMS.drop_duplicates(subset=['chainage', 'lat_round', 'lon_round'], keep='first')
df_unique.info()
df_unique.reset_index(drop=True, inplace=True)
df_unique
#Filtering on bridges only
df_unique['model_type'] = 'bridge'

export_df = df_unique[['road', 'LRPName', 'model_type', 'name', 'lat', 'lon', 'length', 'condition', 'chainage']]
export_df.rename(columns={'LRPName': 'id'}, inplace=True)
export_df.head(10)

In [ ]:
df_proc['model_type'] = 'sourcesink'
df_proc['length'] = 0
df_proc['condition'] = None

df_proc = df_proc[['road', 'lrp', 'model_type', 'name', 'lat', 'lon', 'length', 'condition', 'chainage']]
df_proc.rename(columns={'lrp': 'id'}, inplace=True)

df_proc

In [ ]:
#Adding all bridges to final dataframe
final_df = pd.DataFrame(columns=['road', 'id', 'model_type', 'name', 'lat', 'lon', 'length', 'condition', 'chainage'])
source = df_proc.iloc[[0]]
sink = df_proc.iloc[[-1]]
final_df = pd.concat([source, export_df, sink], ignore_index=True)
final_df = final_df.sort_values('chainage').reset_index(drop=True)
final_df
#Filtering the names in final_df with the help of a regex function that processes the string until the first '('. If after replacing dots, lowercase and strip, the names are exactly the same for two bridges, the bridge is treated as duplicate and is deleted.
final_df = (
    final_df
    .assign(
        prefix=(
            final_df["name"]
            .str.replace('.', '')
            .str.lower()
            .str.strip()
            .str.extract(r"^([^(]+)", expand=False)
        )
    )
    .drop_duplicates(subset="prefix", keep="first")
    .drop(columns="prefix")
)

final_df.head(20)

In [ ]:
    #Adding the links in between the bridges. Their exact locations do not matter and are calculated based on the location of the pre- and succeeding bridge. The length of the link does matter, and is calculated based on the chainage of the two bridges it lies in between.
final_df_with_links = []

for j in range(len(final_df)):
    row = final_df.iloc[j]

    if row['model_type'] == 'source':
        final_df_with_links.append(row)
        continue

    if row['model_type'] == 'sink':
        final_df_with_links.append(row)
        break

    prev = final_df.iloc[j - 1]

    lat_link = (row['lat'] + prev['lat']) / 2
    lon_link = (row['lon'] + prev['lon']) / 2

    link_length = (row['chainage'] - prev['chainage']) * 1000  #Because chainage is in km

    link = {
        'road': row['road'],
        'id': None,
        'model_type': 'link',
        'name': 'Link',
        'lat': lat_link,
        'lon': lon_link,
        'length': link_length,
        'condition': None,
        'chainage': (row['chainage'] + prev['chainage']) / 2
    }

    final_df_with_links.append(pd.Series(link))
    final_df_with_links.append(row)

final_df_with_links = (
    pd.DataFrame(final_df_with_links)
    .reset_index(drop=True)
    .drop(columns='chainage')
)

final_df_with_links['id'] = 100000 + (final_df_with_links.index + 1)

final_df_with_links.head(20)

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(final_df_with_links['lon'], final_df_with_links['lat'])
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(final_df_with_links['lon'], final_df_with_links['lat'], marker='o')
plt.show()
final_df_with_links.to_csv('../data/data_N1.csv', na_rep="None")